# 06. Comparative Evaluation, Robustness & Error Analysis

## Methodological Framing
We evaluate rankers on the exact same test records. We report Primary Chronological Evaluation and Secondary Client-Grouped Robustness independently. Metrics are never averaged or combined across these distinct evaluation paradigms.

### Objectives:
1. Compute Precision@K and Lift over baseline across cutoffs ($K \in [10, 25, 50, 100]$).
2. Execute secondary client-grouped robustness evaluation (zero client overlap).
3. Compute permutation feature importance.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

root_dir = Path.cwd().parent if Path.cwd().name == "work" else Path.cwd()
sys.path.insert(0, str(root_dir))

from src.config import load_config
from src.data import load_dataset
from src.features import build_feature_table
from src.labels import construct_operational_label
from src.splits import create_chronological_split, create_client_grouped_split
from src.baseline import HeuristicRanker
from src.model import ModelPipeline
from src.evaluation import evaluate_rankers
from src.explainability import compute_permutation_importance

config = load_config()

## Step 1: Run Comparative Ranker Evaluation on Test Split

In [ ]:
try:
    df = load_dataset(config.raw_data_path, config)
    is_eligible, y_target, diag = construct_operational_label(df, config.label)
    df_el = df.loc[is_eligible].copy()
    y_el = y_target.loc[is_eligible].copy()
    X, f_names = build_feature_table(df_el, config.schema_mapping.get("features", []))
    
    split = create_chronological_split(df_el, config=config.validation)
    test_idx = split.test_indices
    
    baseline = HeuristicRanker()
    rf = ModelPipeline(model_type="random_forest", random_seed=config.random_seed)
    rf.fit(X.iloc[split.train_indices], y_el.iloc[split.train_indices])
    
    base_scores = baseline.predict_score(X.iloc[test_idx])
    rf_scores = rf.predict_score(X.iloc[test_idx])
    
    results = evaluate_rankers(y_el.iloc[test_idx], base_scores, rf_scores, config.evaluation.k_values)
    print("Primary Chronological Results:")
    display(pd.DataFrame(results["metrics_by_k"]).T)
except FileNotFoundError:
    print("[STATUS: AWAITING REAL DATA EXECUTION] - Empirical metrics render upon warehouse execution.")